In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import re
from os import listdir

from matplotlib import pyplot as plt
from matplotlib import dates as mdates

from ml_project.data import clean_data
from ml_project.feature_engineering import (
    generate_features, impute_missing_data,
    one_hot_encoding, remove_outliers,
    scaling
)
from ml_project.utils import get_project_directories

In [ ]:
directory_paths_dict = get_project_directories()

In [ ]:
sensors_df_raw = pd.read_csv(directory_paths_dict['data_raw'] / "sensors_measurement_warsaw.csv")
sensors_meta_df_raw = pd.read_csv(directory_paths_dict['data_raw'] / "sensors_metadata_warsaw.csv")
locations_df_raw = pd.read_csv(directory_paths_dict['data_raw'] / "locations_dataset_warsaw.csv")
weather_df_raw = pd.read_csv(directory_paths_dict['data_raw'] / "weather_daily_warsaw.csv")

cities_df_raw = pd.read_csv(directory_paths_dict['data_raw'] / "cities.csv")

In [ ]:
city_regex = r"sensors_measurement_(.+)\.csv"
city_names = city_names = [
    re.search(city_regex, f).group(1)
    for f in listdir(directory_paths_dict['data_raw'])
    if re.search(city_regex, f)
]

all_cleaned_dfs = []
for city in city_names:
    data_dir = directory_paths_dict['data_raw']

    try:
        # Load all raw files for this city
        sensors_df_raw = pd.read_csv(data_dir / f"sensors_measurement_{city}.csv")
        sensors_meta_df_raw = pd.read_csv(data_dir / f"sensors_metadata_{city}.csv")
        locations_df_raw = pd.read_csv(data_dir / f"locations_dataset_{city}.csv")
        weather_df_raw = pd.read_csv(data_dir / f"weather_daily_{city}.csv")
        
        # Load shared cities file (same for all)
        cities_df_raw = pd.read_csv(data_dir / "cities.csv")
        
        # Clean data for this city
        cleaned_df = clean_data(
            sensors_df=sensors_df_raw,
            weather_df=weather_df_raw,
            sensors_meta_df=sensors_meta_df_raw,
            locations_df=locations_df_raw,
            cities_df=cities_df_raw
        )
        
        all_cleaned_dfs.append(cleaned_df)
    except FileNotFoundError as e:
        print(f"✗ Skipping {city}: {e}")
    except Exception as e:
        print(f"✗ Error processing {city}: {e}")

pivoted_data_df = pd.concat(all_cleaned_dfs, ignore_index=True)
pivoted_data_df.head()

In [ ]:
out_path = directory_paths_dict["data_processed"] / "daily_pivot.csv"
pivoted_data_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

In [ ]:
pivoted_data_df.describe()